In [1]:
%load_ext autoreload
%autoreload 2
%time

CPU times: user 1 µs, sys: 1 µs, total: 2 µs
Wall time: 4.77 µs


In [2]:
import sys, os
sys.path.append("../../")  # if needed
import pandas as pd
import data_loading as dl

from numu_tki.cc0pi_analyzer import apply_ccnp0pi_stv


In [3]:
RUN = ["1"]
blinded = True

rundata, mc_weights, data_pot = dl.load_runs(
    RUN,
    data="bnb",
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=False,
    loadnumuvariables=True,
    use_bdt=True,
    load_lee=False,
    load_nue_tki=False,
    load_numu_tki=True,
    blinded=blinded,
    enable_cache=True,
)


Loading run 1


In [4]:
import xgboost as xgb
MODEL = "/exp/uboone/app/users/rlalnunt/PELEE_BDT/cc0pi_fstrack_pid_softprob.json"
assert os.path.isfile(MODEL), f"Model file not found: {MODEL}"
print("Using model:", MODEL)
print("xgboost version:", xgb.__version__)

Using model: /exp/uboone/app/users/rlalnunt/PELEE_BDT/cc0pi_fstrack_pid_softprob.json
xgboost version: 1.6.2


In [5]:
mc = rundata["mc"].copy()

In [47]:
import numpy as np

In [48]:
# #Analysis macro for use in the CCNp0pi single transverse variable analysis
# #Re-using selection developed by Steven Gardiner <gardiner@fnal.gov> 22 April 2023
# # Author: Ralte Lalnuntluanga 28 September 2025


from __future__ import annotations
import math
import numpy as np
import pandas as pd

# xgboost is used like the C++ Booster
try:
    import xgboost as xgb
except Exception:
    xgb = None

# ===== constants =====
LOW_FLOAT = -1e30
BOGUS_INDEX = -1

# PDG
ELECTRON_NEUTRINO = 12
MUON = 13
MUON_NEUTRINO = 14
TAU_NEUTRINO = 16
PROTON = 2212
NEUTRON = 2112
PI_ZERO = 111
PI_PLUS = 211

# masses (GeV)
TARGET_MASS = 37.215526
NEUTRON_MASS = 0.93956541
PROTON_MASS = 0.93827208
MUON_MASS = 0.10565837
PI_PLUS_MASS = 0.13957000
BINDING_ENERGY = 0.02478

# cuts
DEFAULT_PROTON_PID_CUT = 0.2
LEAD_P_MIN_MOM_CUT = 0.250
LEAD_P_MAX_MOM_CUT = 1.000
MUON_P_MIN_MOM_CUT = 0.100
MUON_P_MIN_WC_MOM_CUT = 0.121
MUON_P_MAX_MOM_CUT = 2.000
CHARGED_PI_MOM_CUT = 0.07
CHARGED_PI_WC_MOM_CUT = 0.161
MUON_MOM_QUALITY_CUT = 0.25

TOPO_SCORE_CUT = 0.15
COSMIC_IP_CUT = 25.0
TRACK_SCORE_CUT = 0.5

# ===== Fiducial Volume (FV) used for vertex + BDT trk_contained (track END) =====
FV_X_MIN, FV_X_MAX = 21.5, 234.85
FV_Y_MIN, FV_Y_MAX = -95.0, 95.0
FV_Z_MIN, FV_Z_MAX = 21.5, 966.8

# ===== Proton Containment Volume (PCV) used for PFParticle STARTS, muon/proton ENDS =====
PCV_X_MIN, PCV_X_MAX = 10.0, 246.35
PCV_Y_MIN, PCV_Y_MAX = -106.5, 106.5
PCV_Z_MIN, PCV_Z_MAX = 10.0, 1026.8

# ===== helper functions=====
def real_sqrt(x: float) -> float:
    return 0.0 if x < 0.0 else math.sqrt(x)

def in_FV(x: float, y: float, z: float) -> bool:
    return (FV_X_MIN < x < FV_X_MAX and
            FV_Y_MIN < y < FV_Y_MAX and
            FV_Z_MIN < z < FV_Z_MAX)

def in_proton_containment_vol(x: float, y: float, z: float) -> bool:
    return (PCV_X_MIN < x < PCV_X_MAX and
            PCV_Y_MIN < y < PCV_Y_MAX and
            PCV_Z_MIN < z < PCV_Z_MAX)

def is_meson_or_antimeson(pdg: int) -> bool:
    abs_pdg = abs(pdg)
    if abs_pdg >= 9900000: return False
    if (abs_pdg // 1000) % 10 != 0: return False     # thousands digit
    if (abs_pdg // 100) % 10 == 0: return False      # hundreds digit
    if 901 <= abs_pdg <= 930: return False
    if abs_pdg in (110, 990, 998, 999, 100): return False
    return True

def _force_list(v):
    # uproot/awkward → python lists; pass through if already list
    try:
        import awkward as ak
    except Exception:
        ak = None
    if isinstance(v, list): return v
    if ak is not None:
        try:
            return ak.to_list(v)
        except Exception:
            return v
    return v

# ===== STV computations (same formulas as C++) =====
def compute_stvs(p3mu: np.ndarray, p3p: np.ndarray):
    # p3mu, p3p are arrays [3]
    delta_pT_vec = p3mu[:2] + p3p[:2]
    delta_pT = np.linalg.norm(delta_pT_vec)

    # DeltaPhiT
    denom = (np.linalg.norm(p3mu[:2]) * np.linalg.norm(p3p[:2]))
    delta_phiT = np.arccos(
        np.clip((-(p3mu[0]*p3p[0] + p3mu[1]*p3p[1])) / denom, -1.0, 1.0)
    ) if denom > 0 else np.nan

    # Delta_alpha_T
    denom2 = (np.linalg.norm(p3mu[:2]) * np.linalg.norm(delta_pT_vec))
    delta_alphaT = np.arccos(
        np.clip((-(p3mu[0]*delta_pT_vec[0] + p3mu[1]*delta_pT_vec[1])) / denom2, -1.0, 1.0)
    ) if denom2 > 0 else np.nan

    Emu = math.sqrt(MUON_MASS**2 + np.dot(p3mu, p3mu))
    Ep  = math.sqrt(PROTON_MASS**2 + np.dot(p3p,  p3p))
    R = TARGET_MASS + p3mu[2] + p3p[2] - Emu - Ep
    mf = TARGET_MASS - NEUTRON_MASS + BINDING_ENERGY
    delta_pL = 0.5*R - (mf*mf + delta_pT*delta_pT)/(2.0*R) if R != 0 else np.nan

    pn = math.sqrt((delta_pL if np.isfinite(delta_pL) else 0.0)**2 + delta_pT**2)

    # components in transverse plane
    zhat = np.array([0.0, 0.0, 1.0])
    xT = np.cross(zhat, p3mu)[:2]
    xT = xT/np.linalg.norm(xT) if np.linalg.norm(xT) > 0 else np.array([np.nan, np.nan])

    yT = (-p3mu[:2])
    yT = yT/np.linalg.norm(yT) if np.linalg.norm(yT) > 0 else np.array([np.nan, np.nan])

    delta_pTx = float(np.dot(xT, delta_pT_vec)) if np.all(np.isfinite(xT)) else np.nan
    delta_pTy = float(np.dot(yT, delta_pT_vec)) if np.all(np.isfinite(yT)) else np.nan

    theta_mu_p = np.arccos(
        np.clip(np.dot(p3mu, p3p) / (np.linalg.norm(p3mu)*np.linalg.norm(p3p)), -1.0, 1.0)
    ) if np.linalg.norm(p3mu) > 0 and np.linalg.norm(p3p) > 0 else np.nan

    return dict(
        delta_pT=delta_pT,
        delta_phiT=delta_phiT,
        delta_alphaT=delta_alphaT,
        delta_pL=delta_pL,
        pn=pn,
        delta_pTx=delta_pTx,
        delta_pTy=delta_pTy,
        theta_mu_p=theta_mu_p,
    )

# ===== BDT classification (same 7 features, same class order) =====
# Classes: 0=Other, 1=Mu, 2=Pi, 3=P
def _load_booster(model_path: str):
    if xgb is None:
        raise RuntimeError("xgboost not installed. `pip install xgboost`")
    booster = xgb.Booster()
    booster.load_model(model_path)
    return booster

def classify_tracks_for_event(booster, row):
    # build features for generation==2 tracks with track_score > 0.5
    gen        = _force_list(row["pfp_generation_v"])
    tscore     = _force_list(row["trk_score_v"])
    trk_dist   = _force_list(row["trk_distance_v"])
    pid_score  = _force_list(row["trk_llr_pid_score_v"])
    chi2_p     = _force_list(row.get("trk_pid_chipr_v", []))
    KE_p       = _force_list(row["trk_energy_proton_v"])
    dirx       = _force_list(row["trk_dir_x_v"])
    diry       = _force_list(row["trk_dir_y_v"])
    dirz       = _force_list(row["trk_dir_z_v"])
    endx       = _force_list(row["trk_sce_end_x_v"])
    endy       = _force_list(row["trk_sce_end_y_v"])
    endz       = _force_list(row["trk_sce_end_z_v"])
    n_trk_dau  = _force_list(row.get("pfp_trk_daughters_v", []))
    n_shr_dau  = _force_list(row.get("pfp_shr_daughters_v", []))
    mom_range  = _force_list(row["trk_range_muon_mom_v"])
    mom_mcs    = _force_list(row["trk_mcs_muon_mom_v"])

    n = len(gen)
    xgb_pid_vec   = [-1]*n
    xgb_score_vec = [[] for _ in range(n)]

    # IMPORTANT: BDT "trk_contained" uses FV at the TRACK END (C++ bugfix vy/vz).
    trk_end_contained = [in_FV(endx[i], endy[i], endz[i]) for i in range(n)]

    # daughters count
    if n_trk_dau and n_shr_dau and len(n_trk_dau)==n and len(n_shr_dau)==n:
        ndaughters = [int(n_trk_dau[i]) + int(n_shr_dau[i]) for i in range(n)]
    else:
        ndaughters = [0]*n

    def _safe(x, default=0.0):
        try:
            xf = float(x)
            return xf if np.isfinite(xf) else default
        except Exception:
            return default

    feature_rows = []
    feature_indices = []
    for i in range(n):
        if int(gen[i]) != 2:  # only direct nu daughters
            continue
        if float(tscore[i]) <= TRACK_SCORE_CUT:
            continue

        prange = float(mom_range[i]) if i < len(mom_range) else np.nan
        pmcs   = float(mom_mcs[i]) if i < len(mom_mcs) else np.nan
        rel    = (pmcs - prange)/prange if (np.isfinite(prange) and prange > 0) else 0.0


        fs = [
            _safe(trk_dist[i], 0.0),
            _safe(pid_score[i], 0.0),
            _safe(chi2_p[i] if i < len(chi2_p) else 0.0, 0.0),
            _safe(KE_p[i], 0.0),
            1.0 if trk_end_contained[i] else 0.0,  # FV here (matches C++)
            float(ndaughters[i]),
            _safe(rel, 0.0),
        ]

        if not all(np.isfinite(val) for val in fs):
            continue
        feature_rows.append(fs)
        feature_indices.append(i)

    if feature_rows:
        dmat = xgb.DMatrix(np.asarray(feature_rows, dtype=np.float32))
        probs = booster.predict(dmat)  # shape [m,4]: other, mu, pi, p
        for idx, prob in zip(feature_indices, probs):
            k = int(np.argmax(prob))
            xgb_pid_vec[idx] = k
            xgb_score_vec[idx] = prob.tolist()

    # counts by class
    counts = {0:0, 1:0, 2:0, 3:0, -1:0}
    for k in xgb_pid_vec:
        counts[k] = counts.get(k, 0) + 1
    return xgb_pid_vec, xgb_score_vec, counts

# ===== per-event selection & observable computation =====
def _pick_muon_candidate(xgb_pid_vec, xgb_score_vec):
    mu_indices = [i for i,k in enumerate(xgb_pid_vec) if k == 1]
    if not mu_indices:
        return BOGUS_INDEX
    if len(mu_indices) == 1:
        return mu_indices[0]
    # tie-breaker: highest muon softprob (index 1)
    best_i, best_s = BOGUS_INDEX, LOW_FLOAT
    for i in mu_indices:
        s = xgb_score_vec[i][1] if xgb_score_vec[i] else LOW_FLOAT
        if s > best_s:
            best_s, best_i = s, i
    return best_i

def _muon_momentum(row, idx, is_contained):
    rng = float(row["trk_range_muon_mom_v"][idx])
    mcs = float(row["trk_mcs_muon_mom_v"][idx])
    if is_contained:
        # flipped-track veto ONLY if branch exists
        if ("trk_bragg_mu_fwd_preferred_v" in row) and ("trk_pid_chimu_v" in row):
            try:
                trk_bragg_mu_fwd = int(_force_list(row["trk_bragg_mu_fwd_preferred_v"])[idx])
                ntracks = int(row.get("n_tracks", len(_force_list(row["trk_len_v"]))))
                chi2_mu = float(_force_list(row["trk_pid_chimu_v"])[idx])
                if ntracks == 1 and trk_bragg_mu_fwd == 0 and chi2_mu > 6.0:
                    return LOW_FLOAT
            except Exception:
                pass
        return rng
    # not contained → MCS with correction
    if mcs < 0.11:
        return LOW_FLOAT
    return mcs - 0.0361*mcs + 0.04

def _first_proton_window_and_contained(row, idx) -> tuple[bool,bool,float]:
    KEp = float(row["trk_energy_proton_v"][idx])
    p_mom = real_sqrt(KEp*KEp + 2.0*PROTON_MASS*KEp)
    in_window = (LEAD_P_MIN_MOM_CUT <= p_mom <= LEAD_P_MAX_MOM_CUT)
    endx = float(row["trk_sce_end_x_v"][idx])
    endy = float(row["trk_sce_end_y_v"][idx])
    endz = float(row["trk_sce_end_z_v"][idx])
    contained = in_proton_containment_vol(endx, endy, endz)  # PCV for proton ends
    return in_window, contained, p_mom

def _find_leading_proton(row, xgb_pid_vec, mu_idx):
    n = len(xgb_pid_vec)
    lead_idx = BOGUS_INDEX
    lead_len = LOW_FLOAT
    for i in range(n):
        if i == mu_idx: continue
        if xgb_pid_vec[i] != 3:  # class P
            continue
        try:
            trk_len = float(row["trk_len_v"][i])
            if not np.isfinite(trk_len) or trk_len <= 0:
                continue
        except Exception:
            continue
        in_window, contained, _ = _first_proton_window_and_contained(row, i)
        if contained and in_window and trk_len > lead_len:
            lead_len = trk_len
            lead_idx = i
    return lead_idx

def _vector_from_dir_and_p(dirx, diry, dirz, p):
    v = np.array([dirx, diry, dirz], dtype=float)
    n = np.linalg.norm(v)
    if n == 0 or not np.isfinite(n) or p <= 0:
        return np.array([LOW_FLOAT, LOW_FLOAT, LOW_FLOAT], dtype=float)
    v = (v / n) * p
    return v

def analyze_event(row, booster):
    # ensure all vector branches are lists (df.apply passes Series)
    for k in [
        "pfp_generation_v","trk_score_v","trk_distance_v","trk_len_v",
        "trk_llr_pid_score_v","trk_pid_chipr_v","trk_energy_proton_v",
        "trk_dir_x_v","trk_dir_y_v","trk_dir_z_v",
        "trk_sce_end_x_v","trk_sce_end_y_v","trk_sce_end_z_v",
        "trk_sce_start_x_v","trk_sce_start_y_v","trk_sce_start_z_v",
        "trk_range_muon_mom_v","trk_mcs_muon_mom_v",
        "pfp_trk_daughters_v","pfp_shr_daughters_v",
        "trk_bragg_mu_fwd_preferred_v","trk_pid_chimu_v"
    ]:
        if k in row:
            row[k] = _force_list(row[k])

    # 0) “no showers” (generation==2 with track_score <= 0.5)
    gen = row["pfp_generation_v"]
    tscore = row["trk_score_v"]
    reco_shower_count = sum(1 for i in range(len(gen)) if int(gen[i])==2 and float(tscore[i])<=TRACK_SCORE_CUT)
    sel_no_reco_showers = (reco_shower_count == 0)

    # 1) numu CC preselection bits
    # Vertex inside FV (C++ uses FV here)
    sel_reco_vertex_in_FV = in_FV(
        float(row["reco_nu_vtx_sce_x"]),
        float(row["reco_nu_vtx_sce_y"]),
        float(row["reco_nu_vtx_sce_z"])
    )
    sel_topo_cut_passed = float(row["topological_score"]) > TOPO_SCORE_CUT
    sel_cosmic_ip_cut_passed = float(row["CosmicIP"]) > COSMIC_IP_CUT

    # pfps start in PCV (check start positions for gen==2) — PCV here
    sx = row["trk_sce_start_x_v"]; sy = row["trk_sce_start_y_v"]; sz = row["trk_sce_start_z_v"]
    sel_pfp_starts_in_PCV = True
    for i in range(len(gen)):
        if int(gen[i]) != 2: continue
        sel_pfp_starts_in_PCV &= in_proton_containment_vol(float(sx[i]), float(sy[i]), float(sz[i]))

    nslice = int(row.get("nslice", 1))
    sel_presel = (nslice == 1 and sel_reco_vertex_in_FV and sel_pfp_starts_in_PCV and sel_topo_cut_passed and sel_no_reco_showers)

    # 2) classify tracks (BDT "trk_contained" already uses FV)
    xgb_pid_vec, xgb_score_vec, counts = classify_tracks_for_event(booster, row) if sel_presel else ([-1]*len(gen), [[] for _ in gen], {k:0 for k in [0,1,2,3,-1]})

    # 3) muon candidate
    mu_idx = _pick_muon_candidate(xgb_pid_vec, xgb_score_vec)
    sel_has_muon_candidate = (mu_idx != BOGUS_INDEX)

    # 4) muon flags (+ momentum) if have candidate
    muon_contained = False
    muon_passed_mom_cuts = False
    muon_passed_wc_mom_cuts = False
    muon_quality_ok = False
    p3mu = np.array([LOW_FLOAT]*3, dtype=float)

    if sel_has_muon_candidate:
        ex = float(row["trk_sce_end_x_v"][mu_idx]); ey = float(row["trk_sce_end_y_v"][mu_idx]); ez = float(row["trk_sce_end_z_v"][mu_idx])
        muon_contained = in_proton_containment_vol(ex, ey, ez)  # PCV for muon end

        mu_p = _muon_momentum(row, mu_idx, muon_contained)
        if not np.isfinite(mu_p) or mu_p <= 0:
            mu_p = LOW_FLOAT

        # momentum quality
        rng = float(row["trk_range_muon_mom_v"][mu_idx])
        mcs = float(row["trk_mcs_muon_mom_v"][mu_idx])
        muon_quality_ok = (rng > 0 and abs(rng - mcs)/rng < MUON_MOM_QUALITY_CUT)

        # cuts
        if MUON_P_MIN_MOM_CUT <= mu_p <= MUON_P_MAX_MOM_CUT:
            muon_passed_mom_cuts = True
        if MUON_P_MIN_WC_MOM_CUT <= mu_p <= MUON_P_MAX_MOM_CUT:
            muon_passed_wc_mom_cuts = True

        # 3-vector
        p3mu = _vector_from_dir_and_p(float(row["trk_dir_x_v"][mu_idx]),
                                      float(row["trk_dir_y_v"][mu_idx]),
                                      float(row["trk_dir_z_v"][mu_idx]),
                                      mu_p)

        # special case: not contained AND cosθ < −0.9 → invalidate (match C++)
        if (not muon_contained) and np.isfinite(p3mu[2]):
            norm = np.linalg.norm(p3mu)
            costh = p3mu[2]/(norm if norm > 0 else 1.0)
            if costh < -0.9:
                p3mu[:] = LOW_FLOAT

    sel_nu_mu_cc = sel_presel and sel_has_muon_candidate

    # 5) proton candidates and leading proton
    has_p_candidate = False
    protons_contained = False
    passed_proton_pid_cut = False
    lead_p_idx = _find_leading_proton(row, xgb_pid_vec, mu_idx)

    # mark global flags from *all* proton candidates in window & contained
    num_p_candidates = 0
    for i, k in enumerate(xgb_pid_vec):
        if i == mu_idx: continue
        if k != 3: continue
        in_win, contained, _ = _first_proton_window_and_contained(row, i)
        if in_win and contained:
            has_p_candidate = True
            protons_contained = True
            passed_proton_pid_cut = True
            num_p_candidates += 1

    lead_p_passed_mom_cuts = False
    p3p = np.array([LOW_FLOAT]*3, dtype=float)
    if lead_p_idx != BOGUS_INDEX:
        in_win, contained, p_mom = _first_proton_window_and_contained(row, lead_p_idx)
        lead_p_passed_mom_cuts = in_win
        p3p = _vector_from_dir_and_p(float(row["trk_dir_x_v"][lead_p_idx]),
                                     float(row["trk_dir_y_v"][lead_p_idx]),
                                     float(row["trk_dir_z_v"][lead_p_idx]),
                                     p_mom)

    # final CCNp0pi decision (exact C++ logic)
    sel_CCNp0pi = (sel_nu_mu_cc and sel_no_reco_showers and
                   muon_passed_mom_cuts and muon_contained and muon_quality_ok and
                   has_p_candidate and passed_proton_pid_cut and protons_contained and
                   lead_p_passed_mom_cuts)

    # CC0pi (aux)
    sel_CC0pi = (sel_nu_mu_cc and sel_no_reco_showers and
                 muon_passed_mom_cuts and
                 (counts.get(2,0) == 0) and (counts.get(0,0) == 0) and (counts.get(-1,0) == 0))
    sel_CC0pi_wc = (sel_nu_mu_cc and sel_no_reco_showers and
                    muon_passed_wc_mom_cuts and
                    (counts.get(2,0) == 0) and (counts.get(0,0) == 0) and (counts.get(-1,0) == 0))

    # 6) STVs (only if we have both 3-vectors)
    have_mu = np.all(np.isfinite(p3mu)) and (p3mu[0] != LOW_FLOAT)
    have_p  = np.all(np.isfinite(p3p))  and (p3p[0]  != LOW_FLOAT)
    stv = compute_stvs(p3mu, p3p) if (have_mu and have_p) else {k: np.nan for k in
        ["delta_pT","delta_phiT","delta_alphaT","delta_pL","pn","delta_pTx","delta_pTy","theta_mu_p"]}

    return dict(
        # selection bits
        sel_presel=sel_presel,
        sel_nu_mu_cc=sel_nu_mu_cc,
        sel_reco_vertex_in_FV=sel_reco_vertex_in_FV,
        sel_topo_cut_passed=sel_topo_cut_passed,
        sel_cosmic_ip_cut_passed=sel_cosmic_ip_cut_passed,
        sel_pfp_starts_in_PCV=sel_pfp_starts_in_PCV,
        sel_no_reco_showers=sel_no_reco_showers,
        sel_has_muon_candidate=sel_has_muon_candidate,
        sel_muon_contained=muon_contained,
        sel_muon_quality_ok=muon_quality_ok,
        sel_muon_passed_mom_cuts=muon_passed_mom_cuts,
        sel_muon_passed_wc_mom_cuts=muon_passed_wc_mom_cuts,
        sel_has_p_candidate=has_p_candidate,
        sel_passed_proton_pid_cut=passed_proton_pid_cut,
        sel_protons_contained=protons_contained,
        sel_lead_p_passed_mom_cuts=lead_p_passed_mom_cuts,
        sel_CCNp0pi=sel_CCNp0pi,
        sel_CC0pi=sel_CC0pi,
        sel_CC0pi_wc=sel_CC0pi_wc,

        # indices
        muon_candidate_idx=mu_idx,
        lead_p_candidate_idx=lead_p_idx,

        # BDT counts
        sel_n_bdt_other=counts.get(0,0),
        sel_n_bdt_muon=counts.get(1,0),
        sel_n_bdt_pion=counts.get(2,0),
        sel_n_bdt_proton=counts.get(3,0),
        sel_n_bdt_invalid=counts.get(-1,0),

        # vectors
        p3_mu_x=p3mu[0], p3_mu_y=p3mu[1], p3_mu_z=p3mu[2],
        p3_lead_p_x=p3p[0], p3_lead_p_y=p3p[1], p3_lead_p_z=p3p[2],

        # STVs
        **stv,

        # store raw BDT outputs
        xgb_pid_vec=xgb_pid_vec,
        xgb_score_vec=xgb_score_vec,
    )

# ===== public API =====


In [53]:
def apply_ccnp0pi_stv(df: pd.DataFrame, model_path: str) -> pd.DataFrame:
    """
    Adds CCNp0pi selection booleans, candidate indices, BDT outputs, p3 vectors,
    and STVs to the given DataFrame. Expects PeLEE-like branches present as
    list-like columns. Returns a *new* df.
    """
    booster = _load_booster(model_path)

    required = [
        "pfp_generation_v","trk_score_v","trk_distance_v","trk_len_v",
        "trk_llr_pid_score_v","trk_energy_proton_v",
        "trk_dir_x_v","trk_dir_y_v","trk_dir_z_v",
        "trk_sce_end_x_v","trk_sce_end_y_v","trk_sce_end_z_v",
        "trk_sce_start_x_v","trk_sce_start_y_v","trk_sce_start_z_v",
        "trk_range_muon_mom_v","trk_mcs_muon_mom_v",
        "topological_score","CosmicIP",
        "reco_nu_vtx_sce_x","reco_nu_vtx_sce_y","reco_nu_vtx_sce_z",
        "nslice",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    out = df.apply(lambda r: analyze_event(r, booster), axis=1)
    return pd.concat([df.reset_index(drop=True), out.reset_index(drop=True)], axis=1)


In [54]:
mc = apply_ccnp0pi_stv(mc, model_path=MODEL)

KeyboardInterrupt: 